## **Part 1: Data Loading & Setup**

**Q1. Spark Session & Data Loading**
* Create a SparkSession
* Load sales_data.csv
* Infer schema and display schema

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

sales_df = spark.read.csv("sales_data.csv", header=True)

sales_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- order_amount: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_timestamp: string (nullable = true)



**Q2. Data Type Conversion**
* Convert order_date to DateType
* Convert order_timestamp to TimestampType

In [14]:
from pyspark.sql.functions import *

sales_df = sales_df.withColumn("order_date", to_date("order_date"))\
                   .withColumn("order_timestamp", to_timestamp("order_timestamp"))

sales_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- order_amount: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)



## Part 2: Aggregate Functions

**Q3. Overall Sales Metrics**

Calculate:

* Total sales amount (sum)
* Average order amount (avg)
* Maximum and minimum order amount

In [15]:
sales_df.agg(sum('order_amount').alias('total_sales'),\
             avg('order_amount').alias('avg_order_amount'),\
             max('order_amount').alias('max_order_amount'),\
             min('order_amount').alias('min_order_amount'))\
             .show(truncate=False)

+-----------+----------------+----------------+----------------+
|total_sales|avg_order_amount|max_order_amount|min_order_amount|
+-----------+----------------+----------------+----------------+
|359000.0   |44875.0         |80000           |20000           |
+-----------+----------------+----------------+----------------+



**Q4. Region-wise Analysis:**

For each region, calculate:

* Total sales
* Average sales
* Order count

In [16]:
sales_df.groupBy('region')\
        .agg(sum('order_amount').alias('total_sales'),\
             avg('order_amount').alias('avg_sales'),\
             count('order_id').alias('order_count'))\
        .show()

+------+-----------+------------------+-----------+
|region|total_sales|         avg_sales|order_count|
+------+-----------+------------------+-----------+
| South|   102000.0|           51000.0|          2|
|  East|   102000.0|           51000.0|          2|
|  West|    28000.0|           28000.0|          1|
| North|   127000.0|42333.333333333336|          3|
+------+-----------+------------------+-----------+



**Q5. Customer Count:**

Number of distinct customers using countDistinct()

In [18]:
from pyspark.sql.functions import countDistinct
sales_df.agg(countDistinct('customer_id').alias('distinct_customer_count')).show()

+-----------------------+
|distinct_customer_count|
+-----------------------+
|                      4|
+-----------------------+



**Q6. Product-wise Aggregation:**

* collect_list(order_amount)
* collect_set(order_amount)

In [22]:
sales_df.groupBy('product')\
        .agg(collect_list('order_amount').alias('order_amount_list'),\
             collect_set('order_amount').alias('order_amount_set'))\
        .show(truncate=False)

+-------+---------------------+---------------------+
|product|order_amount_list    |order_amount_set     |
+-------+---------------------+---------------------+
|Laptop |[75000, 80000, 72000]|[75000, 80000, 72000]|
|Mobile |[30000, 28000, 32000]|[32000, 30000, 28000]|
|Tablet |[20000, 22000]       |[22000, 20000]       |
+-------+---------------------+---------------------+



## Part 3: Window Functions – Ranking

**Q7. Regional Ranking:**

For each region,
* Assign row_number() based on highest order amount
* Assign rank() and dense_rank()

In [27]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy('region').orderBy(col('order_amount').desc())

sales_df.withColumn('row_number', row_number().over(window_spec))\
        .withColumn('rank', rank().over(window_spec))\
        .withColumn('dense_rank', dense_rank().over(window_spec))\
        .show()

+--------+-----------+------+-------+------------+----------+-------------------+----------+----+----------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|row_number|rank|dense_rank|
+--------+-----------+------+-------+------------+----------+-------------------+----------+----+----------+
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|         1|   1|         1|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|         2|   2|         2|
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|         1|   1|         1|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|         2|   2|         2|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|         3|   3|         3|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|         1|   1|         1|
|    1002|       C0

**Q8. Top Orders per Region**

* Find the top 2 highest orders per region using window functions.

In [32]:
sales_df.withColumn('dense_rank', dense_rank().over(window_spec))\
        .where(col('dense_rank') <= 2).show()

+--------+-----------+------+-------+------------+----------+-------------------+----------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|dense_rank|
+--------+-----------+------+-------+------------+----------+-------------------+----------+
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|         1|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|         2|
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|         1|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|         2|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|         1|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|         2|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|         1|
+--------+-----------+------+-------+------------+----------+---------

## Part 4: Window Functions – Analytical

**Q9. Order Comparison per Customer**
For each customer:
* Use lag() to show previous order amount
* Use lead() to show next order amount

In [36]:
window_spec_cust = Window.partitionBy('customer_id').orderBy('order_timestamp')

sales_df.withColumn('previous_order_amount', lag('order_amount').over(window_spec_cust))\
        .withColumn('next_order_amount', lead('order_amount').over(window_spec_cust))\
        .show()

+--------+-----------+------+-------+------------+----------+-------------------+---------------------+-----------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|previous_order_amount|next_order_amount|
+--------+-----------+------+-------+------------+----------+-------------------+---------------------+-----------------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|                 NULL|            20000|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|                75000|            32000|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|                20000|             NULL|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|                 NULL|            72000|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|                30000|             NULL|
|    1004|       C003|  

**Q10. Running Total**

Calculate running total (cumulative sum) of order amount:
* Partition by customer_id
* Order by order_date

In [38]:
window_spec_cust_2 = Window.partitionBy('customer_id').orderBy('order_date')\
                           .rowsBetween(Window.unboundedPreceding, Window.currentRow)

sales_df.withColumn('running_total', sum('order_amount').over(window_spec_cust_2))\
        .show()

+--------+-----------+------+-------+------------+----------+-------------------+-------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|running_total|
+--------+-----------+------+-------+------------+----------+-------------------+-------------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|      75000.0|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|      95000.0|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|     127000.0|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|      30000.0|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|     102000.0|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|      80000.0|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|     102000.0|
|    1005|       C004|  West| Mobile|   

## Part 5: Window Functions – Aggregates

**Q11. Average Order per Customer**
* Add a column avg_customer_order showing the average order amount per customer using window functions

In [39]:
window_spec_avg = Window.partitionBy('customer_id')

sales_df = sales_df.withColumn('avg_customer_order', avg('order_amount').over(window_spec_avg))

sales_df.show()

+--------+-----------+------+-------+------------+----------+-------------------+------------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|avg_customer_order|
+--------+-----------+------+-------+------------+----------+-------------------+------------------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|42333.333333333336|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|42333.333333333336|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|42333.333333333336|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|           51000.0|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|           51000.0|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|           51000.0|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|          

**Q12. Maximum Order per Region**
* Add a column max_region_order showing the maximum order amount within each region

In [41]:
window_spec_max = Window.partitionBy('region')

sales_df = sales_df.withColumn('max_region_order', max('order_amount').over(window_spec_max))

sales_df.show()

+--------+-----------+------+-------+------------+----------+-------------------+------------------+----------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|avg_customer_order|max_region_order|
+--------+-----------+------+-------+------------+----------+-------------------+------------------+----------------+
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|           51000.0|           80000|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|           51000.0|           80000|
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|42333.333333333336|           75000|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|42333.333333333336|           75000|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|42333.333333333336|           75000|
|    1002|       C002| South| Mobile|       30000|2023-0

## Part 6: Date & Timestamp Functions

**Q13. Date Components Extraction:**
* Year, Month, Day from order_date

In [43]:
sales_df.select(col('order_id'), col('customer_id'), col('order_date'),\
                year(col('order_date')), month(col('order_date')), day(col('order_date'))).show()

+--------+-----------+----------+----------------+-----------------+---------------+
|order_id|customer_id|order_date|year(order_date)|month(order_date)|day(order_date)|
+--------+-----------+----------+----------------+-----------------+---------------+
|    1001|       C001|2023-01-10|            2023|                1|             10|
|    1002|       C002|2023-01-12|            2023|                1|             12|
|    1003|       C001|2023-02-05|            2023|                2|              5|
|    1004|       C003|2023-02-20|            2023|                2|             20|
|    1005|       C004|2023-03-01|            2023|                3|              1|
|    1006|       C002|2023-03-15|            2023|                3|             15|
|    1007|       C001|2023-03-18|            2023|                3|             18|
|    1008|       C003|2023-04-02|            2023|                4|              2|
+--------+-----------+----------+----------------+---------------

**Q14. Date Difference:**
* Days between today and order_date using datediff()

In [45]:
sales_df.select(col('order_id'),\
                col('customer_id'),\
                col('order_date'),\
                date_diff(current_date(), col('order_date'))).show()

+--------+-----------+----------+-------------------------------------+
|order_id|customer_id|order_date|date_diff(current_date(), order_date)|
+--------+-----------+----------+-------------------------------------+
|    1001|       C001|2023-01-10|                                 1171|
|    1002|       C002|2023-01-12|                                 1169|
|    1003|       C001|2023-02-05|                                 1145|
|    1004|       C003|2023-02-20|                                 1130|
|    1005|       C004|2023-03-01|                                 1121|
|    1006|       C002|2023-03-15|                                 1107|
|    1007|       C001|2023-03-18|                                 1104|
|    1008|       C003|2023-04-02|                                 1089|
+--------+-----------+----------+-------------------------------------+



**Q15. Create new columns:**
* order_year
* order_month
* order_week

In [48]:
sales_df = sales_df.withColumn('order_year', year(col('order_date')))\
                   .withColumn('order_month', month(col('order_date')))\
                   .withColumn('order_week', weekofyear(col('order_date')))

sales_df.show()

+--------+-----------+------+-------+------------+----------+-------------------+------------------+----------------+----------+-----------+----------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|avg_customer_order|max_region_order|order_year|order_month|order_week|
+--------+-----------+------+-------+------------+----------+-------------------+------------------+----------------+----------+-----------+----------+
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|           51000.0|           80000|      2023|          2|         8|
|    1008|       C003|  East| Tablet|       22000|2023-04-02|2023-04-02 12:00:00|           51000.0|           80000|      2023|          4|        13|
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|42333.333333333336|           75000|      2023|          1|         2|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|42333.3

**Q16. Date-based Filtering:**
* Placed in March 2023

In [53]:
sales_df.where((col('order_month') == 3) & (col('order_year') == 2023)).show()

+--------+-----------+------+-------+------------+----------+-------------------+------------------+----------------+----------+-----------+----------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|avg_customer_order|max_region_order|order_year|order_month|order_week|
+--------+-----------+------+-------+------------+----------+-------------------+------------------+----------------+----------+-----------+----------+
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|42333.333333333336|           75000|      2023|          3|        11|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|           51000.0|           72000|      2023|          3|        11|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|           28000.0|           28000|      2023|          3|         9|
+--------+-----------+------+-------+------------+----------+-------------------+-------

## Part 7: Real-World ETL Scenarios

**Q17. Monthly Sales Trend:**
* Calculate monthly total sales (group by Year + Month)

In [55]:
sales_df.groupBy('order_year', 'order_month')\
        .agg(sum('order_amount').alias('total_sales'))\
        .orderBy('order_year', 'order_month').show()

+----------+-----------+-----------+
|order_year|order_month|total_sales|
+----------+-----------+-----------+
|      2023|          1|   105000.0|
|      2023|          2|   100000.0|
|      2023|          3|   132000.0|
|      2023|          4|    22000.0|
+----------+-----------+-----------+



**Q18. Customer Activity Analysis:**
* Identify customers who have placed more than 2 orders

In [56]:
sales_df.groupBy('customer_id')\
        .agg(count('order_id').alias('order_count')).where(col('order_count') > 2).show()

+-----------+-----------+
|customer_id|order_count|
+-----------+-----------+
|       C001|          3|
+-----------+-----------+

